# Detecting noisy monitors

This notebook shows how to use the WhyLabs Monitor Diagnoser to customize the diagnosis of a noisy monitor. It interacts with the diagnoser to get information on noisy and failing monitors, and to make selections about which monitor, segment and columns to diagnose.

## Install requirements

In [20]:
#%pip install whylabs-toolkit[diagnoser]


## Setup whylabs API connection

First, set up the information to connect to WhyLabs. Update the org_id, dataset_id and api_key in the following before running it.


In [21]:
import getpass
from whylabs_toolkit.monitor.diagnoser.helpers.utils import env_setup

org_id = input("Enter org ID")
dataset_id = input("Enter model/dataset ID")
api_key = getpass.getpass("Enter API key")
api_endpoint = 'https://api.whylabsapp.com'

env_setup(
    org_id=org_id,
    dataset_id=dataset_id,
    api_key=api_key,
    whylabs_endpoint=api_endpoint
)

Then initialize the Monitor Diagnoser with the org_id and dataset_id.

In [22]:
from whylabs_toolkit.monitor.diagnoser.monitor_diagnoser import MonitorDiagnoser
diagnoser = MonitorDiagnoser(org_id, dataset_id)

# Running a customized diagnosis
## Get the recommended diagnostic interval

Get the dataset start/end time, granularity, and a recommended diagnostic interval for the dataset. The diagnoser will use this interval unless you override it by setting the `diagnostic_interval` property.

In [23]:
lineage, granularity, interval = diagnoser.choose_dataset_batches()
lineage, granularity, interval

(TimeRange(start=datetime.datetime(2021, 5, 20, 0, 0, tzinfo=datetime.timezone.utc), end=datetime.datetime(2024, 5, 13, 21, 0, tzinfo=datetime.timezone.utc)),
 <Granularity.daily: 'daily'>,
 '2024-04-13T00:00:00.000Z/2024-05-13T00:00:00.000Z')

## Get information on noisy and failing monitors

Get information on how many anomalies are detected by each monitor in the dataset. The results are ordered so that the monitors with the most anomalies per column are first (i.e. monitors which are firing on the many batches for certain columns). Beyond that, results with a higher average number of anomalies per column are considered noisier.

In [24]:
import pandas as pd
noisy_monitors = diagnoser.detect_noisy_monitors()
noisy_monitors_df = pd.DataFrame.from_records([m.dict() for m in noisy_monitors])
noisy_monitors_df

,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,frequent-items-drift-monitor-bx6m80,frequent-items-drift-analyzer-bx6m80,frequent_items,3,1,38,30,2,12,0,[]
1,discrete-distribution-22ef37c9-monitor,discrete-distribution-22ef37c9,frequent_items,3,1,38,30,2,12,0,[]
2,smoggy-chartreuse-owl-3387,smoggy-chartreuse-owl-3387-analyzer,frequent_items,3,1,38,30,2,12,0,[]
3,frequent-items-drift-monitor-mat0jo,frequent-items-drift-analyzer-mat0jo,frequent_items,3,1,38,30,2,12,2,"[email, slack]"
4,frequent-items-drift-monitor-01rbfl,frequent-items-drift-analyzer-01rbfl,frequent_items,3,1,38,30,2,12,1,[email]
5,frequent-items-drift-monitor-0foigt,frequent-items-drift-analyzer-0foigt,frequent_items,3,1,38,30,2,12,0,[]
6,frequent-items-drift-monitor-3c0hc2,frequent-items-drift-analyzer-3c0hc2,frequent_items,3,1,38,30,2,12,1,[email]
7,frequent-items-drift-monitor-9gmtix,frequent-items-drift-analyzer-9gmtix,frequent_items,3,1,38,30,2,12,1,[email]
8,elated-palegreen-jaguar-6432,elated-palegreen-jaguar-6432-analyzer,histogram,8,1,81,25,2,10,0,[]
9,frequent-items-drift-monitor-x2hr9z,frequent-items-drift-analyzer-x2hr9z,frequent_items,3,1,22,19,1,7,1,[email]


Once you have run `detect_noisy_monitors`, you can retrieve the result at any time via the `noisy_monitors` property. You can also retrieve
 information about monitors with analysis failures using `failed_monitors`. 

In [25]:
failed_monitors_df = pd.DataFrame.from_records([n.dict() for n in diagnoser.failed_monitors])
failed_monitors_df

,monitor_id,analyzer_id,metric,failed_count,max_failed_per_column,min_failed_per_column,avg_failed_per_column,action_count,action_targets
0,inferred-data-type-fec5a735-monitor,inferred-data-type-fec5a735,inferred_data_type,3,3,3,3,2,"[email, slack]"
1,missing-values-ratio-35881327-monitor,missing-values-ratio-35881327,count_null_ratio,1,1,1,1,0,[]
2,unique-ratio-b7b84aee-monitor,unique-ratio-b7b84aee,unique_est_ratio,1,1,1,1,0,[]


From this information, the diagnoser chooses the most noisy monitor that has notification actions to diagnose. This choice can be overridden by setting the `monitor_id_to_diagnose` property of the diagnoser to the desired monitor id. 

In [26]:
diagnoser.monitor_id_to_diagnose

'frequent-items-drift-monitor-bx6m80'

We can get the monitor object from the diagnoser, to see its display name and any other useful information.

In [27]:
diagnoser.monitor_to_diagnose

Monitor(metadata=Metadata(version=1, schemaVersion=1, updatedTimestamp=1711135588156, author='user_809f777d_3741_4991_8ced_42f09b883ac7', description=None), id='frequent-items-drift-monitor-bx6m80', displayName=None, tags=None, analyzerIds=['frequent-items-drift-analyzer-bx6m80'], schedule=ImmediateSchedule(type='immediate'), disabled=False, severity=3, mode=DigestMode(type='DIGEST', filter=None, creationTimeOffset=None, datasetTimestampOffset=None, groupBy=None), actions=[])

We can similarly see the configuration of the analyzer that is being diagnosed.


In [28]:
diagnoser.analyzer_to_diagnose

Analyzer(metadata=Metadata(version=3, schemaVersion=1, updatedTimestamp=1715716552766, author='user_c9292ec40407f7b580f0a2c90745ebfba2b9e6ea81c848ef944d31e48a45f98', description=None), id='frequent-items-drift-analyzer-bx6m80', displayName=None, tags=None, schedule=FixedCadenceSchedule(type='fixed', cadence=<Cadence.daily: 'daily'>, exclusionRanges=None), disabled=None, disableTargetRollup=None, targetMatrix=ColumnMatrix(segments=[Segment(tags=[])], type=<TargetLevel.column: 'column'>, include=[<ColumnGroups.group_discrete: 'group:discrete'>], exclude=['url', <ColumnGroups.group_output: 'group:output'>, 'desc', 'issue_d'], profileId=None), dataReadinessDuration=None, batchCoolDownPeriod=None, backfillGracePeriodDuration=None, config=DriftConfig(schemaVersion=None, params=None, metric=<ComplexMetrics.frequent_items: 'frequent_items'>, type=<AlgorithmType.drift: 'drift'>, algorithm='hellinger', threshold=0.7, minBatchSize=1, baseline=TrailingWindowBaseline(datasetId=None, inheritSegment=

## Get information on noisy and failing segments in the analyzer

Now we use the diagnoser to get information about noisy and failing segments in the analyzer, so we can choose a segment to diagnose. The results are sorted so the segment with the most anomalies for the selected monitor is first.

In [29]:
from whylabs_toolkit.monitor.diagnoser.helpers.utils import segment_as_readable_text

noisy_segments = diagnoser.detect_noisy_segments()
noisy_segments_df = pd.DataFrame.from_records([n.dict() for n in noisy_segments])
noisy_segments_df['segment'] = [segment_as_readable_text(n.segment.tags) for n in noisy_segments]
noisy_segments_df

,segment,total_anomalies,batch_count
0,overall,38,30


The diagnoser chooses the noisiest segment to diagnose. This can be changed by setting the `diagnostic_segment` property.

In [30]:
segment_as_readable_text(diagnoser.diagnostic_segment.tags)

'overall'

## Get information on noisy columns 

The next step is to get information on the noisy columns within the segment, so we can choose a subset of columns to diagnose. 

In [31]:
noisy_columns = diagnoser.detect_noisy_columns()
noisy_columns_df = pd.DataFrame.from_records([n.dict() for n in noisy_columns])
noisy_columns_df

,column,total_anomalies
0,issue_d,30
1,url,6
2,desc,2
3,disbursement_method,0
4,earliest_cr_line,0
5,emp_length,0
6,emp_title,0
7,grade,0
8,hardship_flag,0
9,home_ownership,0


The API limits diagnosis to 100 columns at a time, so we choose the top 100 noisy columns. We could then iterate through other columns if desired.

In [32]:
columns = list(noisy_columns_df.column[:100])
columns

['issue_d',
 'url',
 'desc',
 'disbursement_method',
 'earliest_cr_line',
 'emp_length',
 'emp_title',
 'grade',
 'hardship_flag',
 'home_ownership',
 'initial_list_status',
 'last_credit_pull_d',
 'last_pymnt_d',
 'loan_status',
 'next_pymnt_d',
 'purpose',
 'pymnt_plan',
 'sub_grade',
 'term',
 'title',
 'verification_status',
 'verification_status_joint',
 'addr_state',
 'zip_code',
 'application_type',
 'debt_settlement_flag']

## Ask for a monitor diagnosis


In [33]:
# for now, we need to enforce this to run using local server
import os
monitor_report = diagnoser.diagnose(columns)

In [34]:
print(monitor_report.describe())

Diagnosis is for monitor "frequent-items-drift-monitor-bx6m80" [frequent-items-drift-monitor-bx6m80] in model-0 org-0, over interval 2024-04-13T00:00:00.000Z/2024-05-13T00:00:00.000Z.

Analyzer is drift configuration for frequent_items metric with TrailingWindow baseline.
Analyzer "frequent-items-drift-analyzer-bx6m80" targets 27 columns and ran on 26 columns in the diagnosed segment.


Diagnostic segment is "overall".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 1517489 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 26 columns and 30 batches.
Found 38 anomalies in 3 columns, with up to 100.0% (30) batches having anomalies per column and 40.0% (12.0) on average.
Columns with anomalies are:
|    | column   |   count |
|---:|:---------|--------:|
|  0 | issue_d  |      30 |
|  1 | url      |       6 |
|  2 | desc     |       2 |

No failures were detected.

Conditions that may impact diagnosis quality include:
	* a